# 🚀 Ray Serve LLM API

This template demonstrates how to deploy **Models** using **Ray Serve + vLLM** and expose it through an **OpenAI-compatible API**.

This a custom template on **Saturn Cloud custom templates** so users can plug-and-play LLM inference environments with GPU acceleration.


## 📦 Install required libraries
Install all the requireed library for the template

In [ ]:
# Install required libraries
!pip install torch transformers fastapi uvicorn "ray[serve, llm]" requests huggingface_hub


## 🧩 Create Ray Serve Deployment File

his writes a file called **`serve_llm.py`** which:

* Configures the model (Qwen2.5-1.5B-Instruct)
* Creates a Ray Serve LLMConfig
* Builds an OpenAI-compatible API using Ray's `build_openai_app`


In [ ]:
%%writefile serve_llm.py
from ray.serve.llm import LLMConfig, build_openai_app

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_ALIAS = "qwen-1.5b"

engine_kwargs = dict(
    tensor_parallel_size=1,
    max_model_len=4096,
)

deployment_config = dict(
    autoscaling_config=dict(
        min_replicas=1,
        max_replicas=1,
    )
)

llm_config = LLMConfig(
    model_loading_config=dict(
        model_id=MODEL_ALIAS,
        model_source=MODEL_ID,
    ),
    engine_kwargs=engine_kwargs,
    deployment_config=deployment_config,
)

app = build_openai_app({"llm_configs": [llm_config]})

## ▶️ Start Ray Serve and Deploy the Model

This will:

* Initialize Ray
* Start Ray Serve
* Deploy the Qwen model as an API at:
  **[http://127.0.0.1:8000/v1/chat/completions](http://127.0.0.1:8000/v1/chat/completions)**

In [ ]:
import ray
from serve_llm import app
from ray import serve

ray.init(ignore_reinit_error=True)

serve.start(detached=False)
serve.run(app)

## 💬 Test the API

Sends a real chat request to your Ray Serve LLM deployment.

In [ ]:
import requests

payload = {
    "model": "qwen-1.5b",
    "messages": [{"role": "user", "content": "Explain API design."}]
}

out = requests.post("http://127.0.0.1:8000/v1/chat/completions", json=payload)
print(out.json())

## ✨ Extract Only the Model 

This grabs the generated text only (no metadata).

In [ ]:
res = out.json()["choices"][0]["message"]["content"]
print(res)

## 🏁 **Conclusion**

You now have a fully running **Ray Serve LLM API** using Qwen2.5-1.5B-Instruct, powered by **vLLM** and exposed through an **OpenAI-compatible endpoint**.
This template can be extended to larger models, added to pipelines, or used inside production-grade ML workloads within Saturn Cloud.

🔗 **Back to Saturn Cloud → [https://saturncloud.io](https://saturncloud.io)**